# Aghashahi Leak Dataset — Feature Extraction (Colab)

Turns the raw Aghashahi et al. (2023) recordings into a **small features CSV** you can
download and upload to the chat.

**How to use:** Runtime → Run all. Then follow the prompt in Step 2.

Works for all three sensor types:
* **Accelerometer** and **Dynamic Pressure Sensor** — `.csv` files @ 51.2 kHz
* **Hydrophone** — `.raw` files @ 8 kHz

Output: one row per 1-second window, ~45 acoustic features + condition labels
(topology, leak type, flow, noise, sensor, recording id).

## Step 1 — Setup

In [ ]:
!pip install -q soundfile
import os, re, sys, glob, zipfile, json, time
import numpy as np, pandas as pd
from scipy import stats, signal as sps
print("ready")

## Step 2 — Get the data

Run the cell. It tries, in order:

1. **Auto-download from Mendeley** (no action needed — takes a few minutes)
2. If that fails, it tells you to use **Option B** (Google Drive) or **Option C** (manual upload),
   which are in the two cells after this one.

In [ ]:
DATA_DIR = "/content/leakdata"
os.makedirs(DATA_DIR, exist_ok=True)

def try_mendeley():
    """Fetch the dataset's file list from the Mendeley API and download the zips."""
    import urllib.request
    api = "https://data.mendeley.com/api/datasets/tbrnp6vrnj/versions/1/files"
    try:
        with urllib.request.urlopen(api, timeout=60) as r:
            files = json.load(r)
    except Exception as e:
        print("Mendeley API not reachable:", e)
        return False
    ok = False
    for f in files:
        name = f.get("filename") or f.get("name", "")
        url  = (f.get("content_details") or {}).get("download_url") or f.get("download_url")
        if not url or not name.lower().endswith(".zip"):
            continue
        dest = os.path.join(DATA_DIR, name)
        if os.path.exists(dest):
            print("already have", name); ok = True; continue
        print("downloading", name, "...", flush=True)
        try:
            urllib.request.urlretrieve(url, dest)
            print("   saved", name, f"({os.path.getsize(dest)/1e6:.0f} MB)")
            ok = True
        except Exception as e:
            print("   failed:", e)
    return ok

got = try_mendeley()
if not got:
    print("\nAuto-download did not work. Use Option B (Drive) or Option C (upload) below.")

### Option B — Google Drive (recommended if auto-download fails)

Put the zips in a Drive folder, then run this cell and edit the path.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp "/content/drive/MyDrive/leak/"*.zip /content/leakdata/
# print(os.listdir(DATA_DIR))

### Option C — Upload a zip straight from your computer (slow for the big ones)

In [ ]:
# from google.colab import files
# up = files.upload()
# for fn in up:
#     os.rename(fn, os.path.join(DATA_DIR, fn))
# print(os.listdir(DATA_DIR))

## Step 3 — Unzip

In [ ]:
for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
    out = os.path.join(DATA_DIR, os.path.splitext(os.path.basename(z))[0])
    if os.path.isdir(out):
        print("already unzipped:", os.path.basename(z)); continue
    print("unzipping", os.path.basename(z), "...", flush=True)
    with zipfile.ZipFile(z) as f:
        f.extractall(out)
print("\nfolders:", [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])

## Step 4 — Inspect the file layout

**Read this cell's output.** It shows the folder structure, example filenames, and the first
lines of one data file. If anything looks unexpected, paste the output into the chat
before running Step 6.

In [ ]:
data_files = []
for root, _, names in os.walk(DATA_DIR):
    for n in names:
        if n.lower().endswith((".csv", ".txt", ".raw")):
            data_files.append(os.path.join(root, n))
print(f"{len(data_files)} data files found\n")

for p in data_files[:8]:
    print("  ", os.path.relpath(p, DATA_DIR))

csvs = [p for p in data_files if p.lower().endswith((".csv", ".txt"))]
if csvs:
    print("\n--- first 3 lines of", os.path.basename(csvs[0]), "---")
    with open(csvs[0], "r", errors="replace") as f:
        for i, line in enumerate(f):
            if i >= 3: break
            print("  ", line.rstrip()[:160])
    probe = pd.read_csv(csvs[0], header=None, nrows=5000, low_memory=False)
    print("\nshape of first 5000 rows:", probe.shape)
    print("dtypes:", probe.dtypes.tolist())

## Step 5 — Feature extraction code

In [ ]:
import os, glob, time
import numpy as np, pandas as pd
from scipy import stats, signal as sps

WIN_SEC, HOP_SEC, N_BANDS = 1.0, 0.5, 16
LEAK_CODES = {"LC", "CC", "GL", "OL", "NL"}

def window_features(w, fs):
    f = {}
    f["rms"]      = float(np.sqrt((w**2).mean()))
    f["log_rms"]  = float(np.log10(f["rms"] + 1e-12))
    f["kurtosis"] = float(stats.kurtosis(w))
    f["skew"]     = float(stats.skew(w))
    f["zcr"]      = float(((w[:-1] * w[1:]) < 0).mean())
    f["peak"]     = float(np.abs(w).max())
    f["crest"]    = float(f["peak"] / (f["rms"] + 1e-12))
    freqs, psd = sps.welch(w, fs=fs, nperseg=min(4096, len(w)))
    psd = psd + 1e-20
    edges, tot = np.linspace(0, fs/2, N_BANDS+1), psd.sum()
    for i in range(N_BANDS):
        m = (freqs >= edges[i]) & (freqs < edges[i+1])
        f[f"band{i:02d}"]    = float(np.log10(psd[m].sum() + 1e-20))
        f[f"bandrel{i:02d}"] = float(psd[m].sum() / tot)      # amplitude-invariant
    p = psd / tot
    f["spec_centroid"]  = float((freqs * p).sum())
    f["spec_spread"]    = float(np.sqrt(((freqs - f["spec_centroid"])**2 * p).sum()))
    f["spec_flatness"]  = float(np.exp(np.log(psd).mean()) / psd.mean())
    f["spec_entropy"]   = float(-(p * np.log(p + 1e-20)).sum())
    f["spec_rolloff85"] = float(freqs[np.searchsorted(np.cumsum(p), 0.85)])
    f["peak_freq"]      = float(freqs[np.argmax(psd)])
    return f

def parse_name(stem):
    """Hydrophone:      LO_GL_0.47 LPS_N_H1 -> topo_leak_flow_noise_sensor (5 parts)
       Accel/pressure:  LO_GL_0.18 LPS_A1   -> topo_leak_flow_sensor       (4 parts,
                        no background-noise condition for these modalities)"""
    p = stem.split("_")
    if len(p) == 5 and p[1] in LEAK_CODES:
        return dict(topo=p[0], leak=p[1], flow=p[2], noise=p[3], sensor=p[4])
    if len(p) == 4 and p[1] in LEAK_CODES:
        return dict(topo=p[0], leak=p[1], flow=p[2], noise="NA", sensor=p[3])
    return None

SR_CANDIDATES = [8000, 12800, 25600, 51200, 102400]

def infer_fs(t, n, default):
    """Sampling rate from the time column: use the full span (robust to the
    low-precision timestamps in these files), then snap to a standard rate."""
    span = t[-1] - t[0]
    if not np.isfinite(span) or span <= 0:
        return default
    raw = (n - 1) / span
    for c in SR_CANDIDATES:
        if abs(raw - c) / c < 0.05:
            return c
    return float(round(raw))

def load_signal(path, fs_hint):
    if path.lower().endswith(".raw"):
        import soundfile as sf
        return sf.read(path, channels=1, samplerate=8000,
                       subtype="PCM_32", endian="LITTLE")[0], 8000
    df = pd.read_csv(path, low_memory=False)
    cols = {str(c).strip().lower(): c for c in df.columns}
    if "value" in cols:                       # the observed "Sample,Value" layout
        sig = pd.to_numeric(df[cols["value"]], errors="coerce").dropna().to_numpy(float)
        if "sample" in cols:
            t = pd.to_numeric(df[cols["sample"]], errors="coerce").dropna().to_numpy(float)
            if len(t) > 1:
                return sig, infer_fs(t, len(sig), fs_hint)
        return sig, fs_hint
    num = df.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all").dropna()
    if num.shape[1] == 0:
        raise ValueError("no numeric columns")
    col = num.iloc[:, -1] if num.shape[1] > 1 else num.iloc[:, 0]
    return col.to_numpy(dtype=float), fs_hint

def extract(folder, out_csv, fs=25600):
    files = sorted(p for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                   if p.lower().endswith((".csv", ".txt", ".raw")))
    print(f"{len(files)} files in {folder}")
    rows, skipped, t0 = [], [], time.time()
    for rid, path in enumerate(files):
        meta = parse_name(os.path.splitext(os.path.basename(path))[0])
        if meta is None:
            skipped.append(os.path.basename(path)); continue
        try:
            sig, fs_used = load_signal(path, fs)
        except Exception as e:
            skipped.append(f"{os.path.basename(path)} ({e})"); continue
        wl, hop = int(WIN_SEC*fs_used), int(HOP_SEC*fs_used)
        if len(sig) < wl:
            skipped.append(f"{os.path.basename(path)} (short)"); continue
        if rid == 0 or (rows and "fs_reported" not in locals()):
            print(f"   sampling rate in use: {fs_used} Hz  "
                  f"({len(sig)} samples = {len(sig)/fs_used:.1f} s)")
            fs_reported = True
        for k, s in enumerate(range(0, len(sig)-wl+1, hop)):
            f = window_features(sig[s:s+wl], fs_used)
            f.update(meta, recording_id=rid, window_idx=k,
                     condition=f"{meta['flow']}|{meta['noise']}",
                     is_leak=int(meta["leak"] != "NL"))
            rows.append(f)
        if rid % 10 == 0:
            pd.DataFrame(rows).to_csv(out_csv, index=False)   # checkpoint
            print(f"  {rid}/{len(files)} files, {len(rows)} windows, "
                  f"{time.time()-t0:.0f}s", flush=True)
    if not rows:
        print("NO WINDOWS EXTRACTED — paste Step 4's output into the chat."); return None
    out = pd.DataFrame(rows)
    out.to_csv(out_csv, index=False)
    print(f"\nwrote {out_csv}: {len(out)} windows from {out.recording_id.nunique()} recordings")
    print("leak/no-leak:", out.is_leak.value_counts().to_dict())
    if skipped:
        print(f"skipped {len(skipped)}, e.g. {skipped[:5]}")
    return out

print("functions defined")

## Step 6 — Run extraction

Edit the folder names below if Step 4 showed different ones. This is the slow part
(roughly 15–30 min for the two big sensor sets); it checkpoints as it goes.

In [ ]:
results = {}
for folder_name, out_name, fs in [
    ("Accelerometer",          "accel_features.csv",    25600),
    ("Dynamic Pressure Sensor","pressure_features.csv", 25600),
    ("Hydrophone",             "hydro_features.csv",     8000),
]:
    matches = [d for d in glob.glob(os.path.join(DATA_DIR, "**"), recursive=True)
               if os.path.isdir(d) and os.path.basename(d) == folder_name]
    if not matches:
        print(f"-- skipping {folder_name} (folder not found)\n"); continue
    print(f"=== {folder_name} ===")
    results[out_name] = extract(matches[0], f"/content/{out_name}", fs)
    print()

## Step 7 — Download the CSVs, then upload them to the chat

In [ ]:
from google.colab import files
for name in ["accel_features.csv", "pressure_features.csv", "hydro_features.csv"]:
    p = f"/content/{name}"
    if os.path.exists(p):
        print(f"{name}: {os.path.getsize(p)/1e6:.1f} MB")
        files.download(p)